# 01 - Data cleaning

First pass over the raw Telco file: what the columns look like, which ones need type fixes,
whether there are duplicates or outliers worth worrying about, and writing out a clean copy that
the rest of the notebooks (and the app) build on.

The actual cleaning logic lives in `src/data.py` so the Streamlit app applies exactly the same
transformations at scoring time.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import CATEGORICAL_COLUMNS, CLEAN_PATH, NUMERIC_COLUMNS, RAW_PATH, clean, load_raw

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [ ]:
raw = load_raw(RAW_PATH)
print(raw.shape)
raw.head()

## Column types

`SeniorCitizen` comes in as 0/1 while every other flag is Yes/No, and `TotalCharges` is read as
`object` even though it is a dollar amount. Everything else is what I expect.

In [ ]:
raw.info()

In [ ]:
raw.isna().sum().sort_values(ascending=False).head()

No missing values are reported by pandas, so the file is complete.

## Categorical values

Checking every categorical column for stray spellings or unexpected levels.

In [ ]:
for col in CATEGORICAL_COLUMNS:
    print(f"{col:18s} {raw[col].unique().tolist()}")

The service columns use `No internet service` / `No phone service` as a third level. That is
redundant with `InternetService == 'No'` / `PhoneService == 'No'`, so the cleaning step folds it
into `No` - it keeps the one-hot encoding smaller and avoids perfectly collinear dummies.

## Duplicates

`customerID` should be unique; checking both the id and full-row duplicates.

In [ ]:
print("duplicate ids :", raw["customerID"].duplicated().sum())
print("duplicate rows:", raw.drop(columns="customerID").duplicated().sum())

No duplicate customers - there are a few rows that are identical apart from the id, which is
plausible for a dataset this size (customers on the same plan with the same tenure), so I keep
them.

## Apply the cleaning

`clean()` parses `TotalCharges` to float, maps `SeniorCitizen` to Yes/No, collapses the redundant
service levels and encodes `Churn` as 0/1.

In [ ]:
df = clean(raw)
df.dtypes

In [ ]:
df[NUMERIC_COLUMNS].describe().round(2)

## Outliers

IQR fences on the three numeric columns. `TotalCharges` is right-skewed simply because it is
tenure x monthly charge, so a long tail is expected there rather than an error.

In [ ]:
def iqr_outliers(s: pd.Series) -> int:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

pd.Series({c: iqr_outliers(df[c]) for c in NUMERIC_COLUMNS}, name="iqr_outliers")

Zero points fall outside the IQR fences, so nothing is clipped or removed. The charge columns
are bounded by the plan catalogue anyway (there is no such thing as a $10,000 monthly plan).

## Target balance

In [ ]:
df["Churn"].value_counts(normalize=True).rename("share").round(3)

About 26.5% of customers churned. That is imbalanced enough that plain accuracy will be a
misleading headline number (predicting "stays" for everyone already scores 73%), which is why the
modelling notebook resamples with SMOTE and reports recall, F1 and ROC-AUC.

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print("saved", CLEAN_PATH.relative_to(PROJECT_ROOT), df.shape)